In [3]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import pygmt

# Grow the GPU memory usage as needed
gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices("GPU")
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

import nn_gmm

2022-03-29 09:02:33.604311: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.1
2022-03-29 09:02:34.745864: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2022-03-29 09:02:34.745914: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2022-03-29 09:02:34.793860: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-03-29 09:02:34.794304: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:08:00.0 name: NVIDIA GeForce RTX 2060 computeCapability: 7.5
coreClock: 1.755GHz coreCount: 30 deviceMemorySize: 5.79GiB deviceMemoryBandwidth: 312.97GiB/s
2022-03-29 09:02:34.794330: I tensorflow/stream_executor/platform/de

1 Physical GPUs, 1 Logical GPUs


In [4]:
# model_dir = Path("/Users/claudy/dev/work/tmp/models/0301_145629_base-grid_fault-weights_XYZ_loc")
# z_ffp = Path("/Users/claudy/dev/work/tmp/site_params.csv")
# map_data_ffp = Path("/Users/claudy/dev/work/code/qcore/qcore/data")

model_dir = Path("/home/claudy/dev/work/data/nn_gmm/results/keep_runs/current/0301_145629_base-grid_fault-weights_XYZ_loc")
z_ffp = Path("/home/claudy/dev/work/storage/data/nn_gmm/input_data/site_params/site_params.csv")
map_data_ffp = Path("/home/claudy/dev/work/code/qcore/qcore/data")

In [5]:
# Load the data and compute the residual and squared residual
columns = ["site", "lat", "lon", "pSA_3.0", "pSA_3.0_est", "event", "vs30"]

data_df = nn_gmm.ResultDB.get_data_static(model_dir / "train_predictions.hdf5", columns=columns)
data_df["res"] = data_df["pSA_3.0"] - data_df["pSA_3.0_est"]
data_df["squared_res"] = data_df["res"].values**2
data_df["realisation"] = np.stack(np.char.rsplit(data_df.index.values.astype(str), "_", maxsplit=1))[:, 0]

# Z-data
site_params = pd.read_csv(z_ffp, index_col=0)

In [6]:
# Sort by worst and create mask for worst
data_df.sort_values("squared_res", ascending=False, inplace=True)
mask = data_df.squared_res > np.quantile(data_df.squared_res, 0.999)

In [7]:
data_df.loc[mask]

,lat,lon,pSA_3.0,vs30,pSA_3.0_est,event,site,res,squared_res,realisation
2016p871889_2016p871889_0200859,-41.510830,172.947968,-5.134186,455.000000,-8.412897,2016p871889_2016p871889_0200859,0200859,3.278711,10.749945,2016p871889_2016p871889
2016p859524_2016p859524_HORC,-43.539600,171.959900,-3.514093,521.475037,-6.782937,2016p859524_2016p859524_HORC,HORC,3.268843,10.685337,2016p859524_2016p859524
Inangahua_REL29_MGCS,-41.507702,173.944397,-1.457009,455.000000,-4.691531,Inangahua,MGCS,3.234522,10.462134,Inangahua_REL29
2016p859524_2016p859524_02006c7,-43.519230,171.959457,-3.507386,505.186035,-6.734549,2016p859524_2016p859524_02006c7,02006c7,3.227162,10.414577,2016p859524_2016p859524
Inangahua_REL29_020099b,-41.506451,173.907578,-1.340581,305.083984,-4.507848,Inangahua,020099b,3.167268,10.031586,Inangahua_REL29
...,...,...,...,...,...,...,...,...,...,...
NorthBranch_REL17_0200597,-44.370419,171.043381,-5.941529,462.206543,-4.334031,NorthBranch,0200597,-1.607498,2.584050,NorthBranch_REL17
MS04_REL26_0200b6f,-41.057968,175.139923,-5.826279,455.000000,-4.218787,MS04,0200b6f,-1.607491,2.584029,MS04_REL26
Riversdale_REL02_0200b30,-39.408020,174.992249,-3.472515,455.000000,-5.079896,Riversdale,0200b30,1.607381,2.583675,Riversdale_REL02
KerepehiO_REL24_0200d7f,-37.234741,175.828171,-4.837265,240.000000,-3.229915,KerepehiO,0200d7f,-1.607350,2.583575,KerepehiO_REL24


In [8]:
# Number of times each station occurs in list above
data_df.loc[mask].groupby("site").count().sort_values("lat", ascending=False)["lat"].rename("count")

site
0200844    191
02006c7    153
0200859    153
MGCS       144
HORC       135
          ... 
02008f9      1
02008fc      1
0200900      1
0200902      1
WVHS         1
Name: count, Length: 1035, dtype: int64

In [9]:
# cur_site = "0200844"
# cur_neighbour = "020081c"
# region = (172.598, 173.043, -42.900, -42.685)


# -43.519230,171.959457
cur_site = "02006c7"
cur_neighbour = "02006a3"
region = (171.8, 172.15, -43.6, -43.43)

In [10]:
# Look event count for current site
site_mask = mask & (data_df.site == cur_site)
data_df.loc[site_mask].groupby("event").count().sort_values("lat", ascending=False)["lat"].rename("count")

event
HopeConwayOS                       28
NMFZB0                             19
Harper                             15
Esk                                11
AlpineK2T                           9
Waitohi                             9
Browning                            8
Ashley                              7
MtWhite                             6
Lowry                               6
HopeCW                              4
HopeTARA                            4
BrunAnt                             3
Torlesse                            2
Poulter                             2
ClarenceNE                          2
KaiwaraS                            2
Inangahua                           2
Springbank                          1
2016p881669_2016p881669_02006c7     1
Maimai                              1
3468672_3468672_02006c7             1
2252479_2252479_02006c7             1
3590805_3590805_02006c7             1
Hope1888                            1
3368445_3368445_02006c7             1
2016p8

In [11]:
# Look at one event for current site
cur_event = "HopeConwayOS"
site_event_mask = site_mask & (data_df.loc[site_mask].event == cur_event)
cur_data = data_df.loc[site_event_mask].sort_index()

# Set index to realisation
cur_data.index = np.stack(np.char.rsplit(cur_data.index.values.astype(str), "_", maxsplit=1))[:, 0]

cur_data

,lat,lon,pSA_3.0,vs30,pSA_3.0_est,event,site,res,squared_res,realisation
HopeConwayOS_REL02,-43.51923,171.959457,-1.813848,505.186035,-3.744585,HopeConwayOS,02006c7,1.930737,3.727746,HopeConwayOS_REL02
HopeConwayOS_REL03,-43.51923,171.959457,-2.544447,505.186035,-4.313961,HopeConwayOS,02006c7,1.769514,3.131179,HopeConwayOS_REL03
HopeConwayOS_REL04,-43.51923,171.959457,-2.442258,505.186035,-4.492101,HopeConwayOS,02006c7,2.049843,4.201855,HopeConwayOS_REL04
HopeConwayOS_REL05,-43.51923,171.959457,-2.492652,505.186035,-4.212401,HopeConwayOS,02006c7,1.719750,2.957539,HopeConwayOS_REL05
HopeConwayOS_REL06,-43.51923,171.959457,-2.349862,505.186035,-4.353334,HopeConwayOS,02006c7,2.003472,4.013900,HopeConwayOS_REL06
HopeConwayOS_REL07,-43.51923,171.959457,-2.407479,505.186035,-4.309330,HopeConwayOS,02006c7,1.901852,3.617041,HopeConwayOS_REL07
HopeConwayOS_REL09,-43.51923,171.959457,-2.747105,505.186035,-4.404064,HopeConwayOS,02006c7,1.656958,2.745511,HopeConwayOS_REL09
HopeConwayOS_REL10,-43.51923,171.959457,-1.707046,505.186035,-3.746141,HopeConwayOS,02006c7,2.039095,4.157907,HopeConwayOS_REL10
HopeConwayOS_REL11,-43.51923,171.959457,-2.162757,505.186035,-4.079506,HopeConwayOS,02006c7,1.916749,3.673927,HopeConwayOS_REL11
HopeConwayOS_REL12,-43.51923,171.959457,-1.291167,505.186035,-3.552035,HopeConwayOS,02006c7,2.260868,5.111522,HopeConwayOS_REL12


In [12]:
# Get data for neighbour station
cur_neighbour_data = data_df.loc[(data_df.site == cur_neighbour) & (data_df.event == cur_event)].sort_index()
cur_neighbour_data.index = np.stack(np.char.rsplit(cur_neighbour_data.index.values.astype(str), "_", maxsplit=1))[:, 0]

cur_neighbour_data

,lat,lon,pSA_3.0,vs30,pSA_3.0_est,event,site,res,squared_res,realisation
HopeConwayOS_REL01,-43.518372,171.860474,-3.176759,455.118774,-3.788010,HopeConwayOS,02006a3,0.611251,0.373628,HopeConwayOS_REL01
HopeConwayOS_REL02,-43.518372,171.860474,-2.829438,455.118774,-3.871780,HopeConwayOS,02006a3,1.042341,1.086475,HopeConwayOS_REL02
HopeConwayOS_REL03,-43.518372,171.860474,-3.795503,455.118774,-4.550578,HopeConwayOS,02006a3,0.755075,0.570138,HopeConwayOS_REL03
HopeConwayOS_REL04,-43.518372,171.860474,-3.625996,455.118774,-4.614749,HopeConwayOS,02006a3,0.988754,0.977634,HopeConwayOS_REL04
HopeConwayOS_REL05,-43.518372,171.860474,-3.553288,455.118774,-4.414194,HopeConwayOS,02006a3,0.860906,0.741159,HopeConwayOS_REL05
HopeConwayOS_REL06,-43.518372,171.860474,-3.175246,455.118774,-4.651100,HopeConwayOS,02006a3,1.475854,2.178144,HopeConwayOS_REL06
HopeConwayOS_REL07,-43.518372,171.860474,-3.547207,455.118774,-4.439235,HopeConwayOS,02006a3,0.892028,0.795714,HopeConwayOS_REL07
HopeConwayOS_REL08,-43.518372,171.860474,-3.860880,455.118774,-4.668468,HopeConwayOS,02006a3,0.807588,0.652199,HopeConwayOS_REL08
HopeConwayOS_REL09,-43.518372,171.860474,-3.832282,455.118774,-4.616599,HopeConwayOS,02006a3,0.784317,0.615153,HopeConwayOS_REL09
HopeConwayOS_REL10,-43.518372,171.860474,-2.671769,455.118774,-3.894373,HopeConwayOS,02006a3,1.222604,1.494760,HopeConwayOS_REL10


In [13]:
# Comparison of site conditions
site_params.loc[[cur_site, cur_neighbour]]

,lon,lat,vs30,vs500,z1p0,z2p5
station,,,,,,
02006c7,171.95946,-43.51923,505.186037,0.446010,1.085,2.125
02006a3,171.86047,-43.51837,455.118765,0.827331,0.175,1.225


In [14]:
# Compute the differences between site of interest and neighbour
shared_realisations = np.intersect1d(cur_data.index.values, cur_neighbour_data.index.values)

diff_data = pd.DataFrame(data=(cur_neighbour_data.loc[shared_realisations, "pSA_3.0"] - cur_data.loc[shared_realisations,  "pSA_3.0"]).values, index=shared_realisations, columns=["ln_ratio"])
diff_data["im_diff"] = np.exp(cur_neighbour_data.loc[shared_realisations, "pSA_3.0"].values) - np.exp(cur_data.loc[shared_realisations,  "pSA_3.0"]).values
diff_data["perc_diff"] = np.exp(cur_neighbour_data.loc[shared_realisations, "pSA_3.0"].values) / np.exp(cur_data.loc[shared_realisations,  "pSA_3.0"]).values

diff_data.sort_values("perc_diff", ascending=False)

,ln_ratio,im_diff,perc_diff
HopeConwayOS_REL33,-0.792167,-0.155136,0.452863
HopeConwayOS_REL14,-0.809570,-0.101751,0.445050
HopeConwayOS_REL06,-0.825385,-0.053599,0.438066
HopeConwayOS_REL26,-0.855337,-0.117251,0.425140
HopeConwayOS_REL35,-0.891665,-0.081474,0.409973
HopeConwayOS_REL18,-0.911344,-0.072748,0.401984
HopeConwayOS_REL25,-0.962273,-0.084846,0.382024
HopeConwayOS_REL10,-0.964723,-0.112271,0.381089
HopeConwayOS_REL02,-1.015590,-0.103980,0.362189
HopeConwayOS_REL17,-1.023533,-0.112087,0.359323


In [15]:
# Load map data & gmt config
map_data = nn_gmm.NZMapData.load(map_data_ffp)

In [ ]:
# Generate a loction plot
fig = nn_gmm.gen_region_fig("Location", region, map_data=map_data, plot_kwargs={"topo_cmap": "oleron"})

# Add site + label
fig.plot(x=site_params.loc[cur_site].lon, y=site_params.loc[cur_site].lat, style="t0.25c", color="white", pen="black")
fig.text(text=cur_site, x=site_params.loc[cur_site].lon, y=site_params.loc[cur_site].lat - 0.05)

fig.plot(x=site_params.loc[cur_neighbour].lon, y=site_params.loc[cur_neighbour].lat, style="t0.25c", color="white", pen="black")
fig.text(text=cur_neighbour, x=site_params.loc[cur_neighbour].lon, y=site_params.loc[cur_neighbour].lat - 0.05)

fig.show()

In [ ]:
# Vs30 map
fig = nn_gmm.gen_region_fig("Vs30", region, map_data=map_data)

grid = nn_gmm.create_grid(site_params, "vs30", region=region)
nn_gmm.plot_grid(fig, grid, "hot", (100, 1000, 900 / 12),
                 ("white", "black"), "Vs30", transparency=25, reverse_cmap=True)

# Add site + label
fig.plot(x=172.85104, y=-42.80441, style="t0.25c", color="white", pen="black")
fig.text(text="0200844", x=172.85104, y=-42.790441)

fig.plot(x=172.75314, y=-42.80434, style="t0.25c", color="white", pen="black")
fig.text(text="020081c", x=172.75314, y=-42.790434)

fig.show()

In [ ]:
# Z1.0 map
fig = nn_gmm.gen_region_fig("Z1.0", region, map_data=map_data)

grid = nn_gmm.create_grid(site_params, "z1p0", region=region)
nn_gmm.plot_grid(fig, grid, "hot", (0.1, 1.5, np.round((1.5 - 0.1) / 12, 3)),
                 ("white", "black"), "Z1.0", transparency=25, reverse_cmap=True)

# Add site + label
fig.plot(x=172.85104, y=-42.80441, style="t0.25c", color="white", pen="black")
fig.text(text="0200844", x=172.85104, y=-42.790441)

fig.plot(x=172.75314, y=-42.80434, style="t0.25c", color="white", pen="black")
fig.text(text="020081c", x=172.75314, y=-42.790434)

fig.show()

In [ ]:
# Z2.5 map
fig = nn_gmm.gen_region_fig("Z2.5", region, map_data=map_data)

grid = nn_gmm.create_grid(site_params, "z2p5", region=region)
nn_gmm.plot_grid(fig, grid, "hot", (0.1, 10.0, (10.0 - 0.1) / 12),
                 ("white", "black"), "Z2.5", transparency=25)

# Add site + label
fig.plot(x=172.85104, y=-42.80441, style="t0.25c", color="white", pen="black")
fig.text(text="0200844", x=172.85104, y=-42.790441)

fig.plot(x=172.75314, y=-42.80434, style="t0.25c", color="white", pen="black")
fig.text(text="020081c", x=172.75314, y=-42.790434)


fig.show()

In [ ]:
cur_realisation = "Harper_REL09"
fig = nn_gmm.im_plot(data_df.loc[data_df.realisation == cur_realisation], "pSA_3.0", cur_realisation, map_data_ffp, region=region)

# Add site + label
fig.plot(x=172.85104, y=-42.80441, style="t0.25c", color="white", pen="black")
fig.text(text="0200844", x=172.85104, y=-42.795441)

fig.plot(x=172.75314, y=-42.80434, style="t0.25c", color="white", pen="black")
fig.text(text="020081c", x=172.75314, y=-42.795434)

fig.show()

print(f"{cur_site}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_site}", "pSA_3.0"]))
print(f"{cur_neighbour}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_neighbour}", "pSA_3.0"]))

In [ ]:
cur_realisation = "Harper_REL14"
fig = nn_gmm.im_plot(data_df.loc[data_df.realisation == cur_realisation], "pSA_3.0", cur_realisation, map_data_ffp, region=region)

# Add site + label
fig.plot(x=172.85104, y=-42.80441, style="t0.25c", color="white", pen="black")
fig.text(text="0200844", x=172.85104, y=-42.795441)

fig.plot(x=172.75314, y=-42.80434, style="t0.25c", color="white", pen="black")
fig.text(text="020081c", x=172.75314, y=-42.795434)

fig.show()

print(f"{cur_site}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_site}", "pSA_3.0"]))
print(f"{cur_neighbour}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_neighbour}", "pSA_3.0"]))

In [ ]:
cur_realisation = "Harper_REL06"
fig = nn_gmm.im_plot(data_df.loc[data_df.realisation == cur_realisation], "pSA_3.0", cur_realisation, map_data_ffp, region=region)

# Add site + label
fig.plot(x=172.85104, y=-42.80441, style="t0.25c", color="white", pen="black")
fig.text(text="0200844", x=172.85104, y=-42.795441)

fig.plot(x=172.75314, y=-42.80434, style="t0.25c", color="white", pen="black")
fig.text(text="020081c", x=172.75314, y=-42.795434)

fig.show()

print(f"{cur_site}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_site}", "pSA_3.0"]))
print(f"{cur_neighbour}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_neighbour}", "pSA_3.0"]))

In [ ]:
cur_realisation = "Browning_REL12"
fig = nn_gmm.im_plot(data_df.loc[data_df.realisation == cur_realisation], "pSA_3.0", cur_realisation, map_data_ffp, region=region)

# Add site + label
fig.plot(x=172.85104, y=-42.80441, style="t0.25c", color="white", pen="black")
fig.text(text="0200844", x=172.85104, y=-42.795441)

fig.plot(x=172.75314, y=-42.80434, style="t0.25c", color="white", pen="black")
fig.text(text="020081c", x=172.75314, y=-42.795434)

fig.show()

print(f"{cur_site}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_site}", "pSA_3.0"]))
print(f"{cur_neighbour}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_neighbour}", "pSA_3.0"]))

In [ ]:
cur_realisation = "Browning_REL03"
fig = nn_gmm.im_plot(data_df.loc[data_df.realisation == cur_realisation], "pSA_3.0", cur_realisation, map_data_ffp, region=region)

# Add site + label
fig.plot(x=172.85104, y=-42.80441, style="t0.25c", color="white", pen="black")
fig.text(text="0200844", x=172.85104, y=-42.795441)

fig.plot(x=172.75314, y=-42.80434, style="t0.25c", color="white", pen="black")
fig.text(text="020081c", x=172.75314, y=-42.795434)

fig.show()

print(f"{cur_site}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_site}", "pSA_3.0"]))
print(f"{cur_neighbour}:", np.exp(data_df.loc[f"{cur_realisation}_{cur_neighbour}", "pSA_3.0"]))

## Distribution of outliers wrt. to IM

In [ ]:
data_df

In [ ]:
im_values = np.exp(data_df["pSA_3.0"].values)

In [ ]:
fig = plt.figure(figsize=(20, 10))

ax = fig.add_subplot(1, 2, 1)
# ax.hist(im_values[im_values < 0.2], bins=50);
ax.hist(data_df.loc[:, "pSA_3.0"].values, bins=50);
ax.set_title("Distribution of pSA_3.0")

ax = fig.add_subplot(1, 2, 2)
ax.hist(data_df.loc[mask, "pSA_3.0"].values, bins=50);
ax.set_title("Distribution of worst 1% of outliers (pSA_3.0)")

In [ ]:
# bins = np.arange(0.0, 0.2, 0.01)
# bin_ind = np.digitize(im_values, bins)

bins = np.arange(-12.0, np.log(0.2), 0.1)
bin_ind = np.digitize(data_df["pSA_3.0"], bins)

df = data_df.copy()
df["bin_ix"] = bin_ind
df["abs_res"] = np.abs(df["res"].values)

group_mean = df.loc[:, ["abs_res", "squared_res", "bin_ix"]].groupby("bin_ix").mean()
abs_res_binned = group_mean.abs_res.values
squared_res_binned = group_mean.squared_res.values

In [ ]:
fig = plt.figure(figsize=(20, 10))

plt.plot(bins, abs_res_binned[:-1], marker=".", label="Mean Absolute Error (per bin)")
plt.plot(bins, squared_res_binned[:-1], marker=".", label="Mean Squared Error (per bin)")
plt.hist(data_df.loc[:, "pSA_3.0"].values, bins=50, density=True);
plt.hist(data_df.loc[mask, "pSA_3.0"].values, bins=50, density=True, alpha=0.5);

plt.legend()